# Training and Evaluation of the Clinical HIV Viral Load Classification Model

This notebook trains and evaluates a convolutional neural network for classifying HIV viral load image sequences into clinically defined viral load categories. The model uses reduced 7-frame tensor inputs derived from microscopy time-series data.

The clinical model uses four ordered classes:

- `undetectable`
- `low`
- `medium`
- `high`

The workflow includes loading reduced tensor datasets, adapting a ResNet-18 architecture for 7-channel inputs, training the model with class-balanced sampling, selecting the best validation model, evaluating performance on a held-out test set, and generating a confusion matrix for model interpretation.

## 1. Import required libraries

This section imports the libraries required for dataset loading, model construction, training, evaluation, and visualization.

PyTorch is used for deep learning model development. Torchvision provides the ResNet-18 backbone. Scikit-learn is used to compute the confusion matrix. Matplotlib and Seaborn are used for figure generation.

In [ ]:
import os
from collections import Counter
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

## 2. Define the clinical tensor dataset

The model is trained using reduced PyTorch `.pt` tensor files. Each tensor represents a 7-frame reduced image sequence derived from the original microscopy time series.

The dataset folder is expected to contain one subfolder per clinical class:

- `undetectable`
- `low`
- `medium`
- `high`

The custom `PTDataset` class scans each class folder, assigns numeric labels based on folder name, loads the tensor files, removes an extra singleton channel dimension when present, and returns `(tensor, label)` pairs for model training and evaluation.

In [ ]:
class PTDataset(Dataset):
    def __init__(self, root_dir, target_size=(500, 500), transform=None):
        """
        Args:
            root_dir (str): Path to the dataset directory (e.g., Training folder).
            target_size (tuple): Desired output size (height, width).
            transform (callable, optional): Optional transformations (on CPU).
        """
        self.root_dir = root_dir
        self.target_size = target_size
        self.transform = transform
        self.classes = ['undetectable', 'low', 'medium', 'high']

         # Collect all file paths and labels
        self.file_list = []
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):
                if file.endswith('.pt'):
                    full_path = os.path.join(class_path, file)
                    class_index = self.classes.index(label)
                    self.file_list.append((full_path, class_index))

        # Pre-load everything into memory
        self.data_list = []
        for file_path, label in self.file_list:
            # Load reduced tensor from disk
            tensor_data = torch.load(file_path, map_location='cpu')

            # Reduced files should already be [1, 7, H, W] or [7, H, W]
            if tensor_data.dim() == 4 and tensor_data.shape[0] == 1:
                tensor_data = tensor_data.squeeze(0)  # [7, H, W]

            # Optional transform
            if self.transform:
                tensor_data = self.transform(tensor_data)

            # Store (tensor, label)
            self.data_list.append((tensor_data, label))

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

## 3. Define the ResNet model and training utilities

This section defines the model architecture and the functions used for training and evaluation.

A ResNet-18 model is used as the classification backbone. Because the input tensors contain 7 temporal channels instead of 3 RGB channels, the first convolutional layer is replaced with a new layer that accepts 7 input channels. The final fully connected layer is replaced with a dropout layer followed by a linear classifier that outputs four clinical viral load categories.

The training loop uses cross-entropy loss for single-label multi-class classification. Validation performance is evaluated after each epoch, and automatic mixed precision is used when CUDA is available to improve GPU training efficiency.

In [ ]:
def get_resnet_model(num_classes=4, input_channels=7, dropout_rate=0.243493213909431):
    """
    Build ResNet18 with a custom first conv layer
    that expects `input_channels` and adds a Dropout layer.

    model_depth = 18 (ResNet18)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Replace first conv to match your input_channels
    model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    # Replace FC layer to include Dropout before classification
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),  # Dropout before final classification
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model


def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Use AMP if on GPU
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=25):
    """
    Basic training routine using CrossEntropyLoss
    for single-label, multi-class classification.
    """
    
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total

        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    print("Training complete.")



## 4. Configure dataset paths and training parameters

This section defines the dataset locations and output paths used by the notebook.

For Dockerized execution, the clinical dataset should be mounted or rebuilt into the following folder structure:

```python
/workspace/data/clinical/Training
/workspace/data/clinical/Validation
/workspace/data/clinical/Testing
```

Each split should contain the four clinical class folders:

```python
undetectable/
low/
medium/
high/
```

The trained model weights and generated outputs are saved to:

```python
/workspace/outputs
```

The model is trained multiple times, and the run with the highest validation accuracy is selected as the final clinical model.

In [ ]:
DATA_ROOT = Path("/home/jovyan/work/data/reduced")
RESULTS_ROOT = Path("/home/jovyan/work/results")
MODEL_ROOT = Path("/home/jovyan/work/models")

MODEL_PATH = MODEL_ROOT / "clinical_resnet18_best.pth"

TRAIN_DIR = DATA_ROOT / "Training"
VAL_DIR = DATA_ROOT / "Validation"
TEST_DIR = DATA_ROOT / "Testing"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

## 5. Train and evaluate the clinical model

The main training routine performs the full model training workflow:

1. Selects CUDA if a GPU is available, otherwise uses CPU.
2. Loads the clinical training, validation, and testing datasets.
3. Computes class-balanced sampling weights from the training set.
4. Trains the ResNet-18 model using fixed hyperparameters.
5. Repeats training across multiple runs.
6. Selects the model with the highest validation accuracy.
7. Saves the best model weights.
8. Evaluates the selected model on the held-out test set.

Class-balanced sampling is used to reduce bias from unequal class frequencies in the training data.

In [ ]:
def main():
    #Check device for CUDA or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    
    train_dataset = PTDataset(root_dir=TRAIN_DIR, target_size=(500, 500))
    val_dataset   = PTDataset(root_dir=VAL_DIR,   target_size=(500, 500))
    test_dataset  = PTDataset(root_dir=TEST_DIR,  target_size=(500, 500))

   
    train_labels = [label for _, label in train_dataset.data_list]
    print("Labels in dataset:", set(train_labels))

    class_counts = Counter(train_labels)
    weights = [1.0 / class_counts[label] for label in train_labels]
    
    train_sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )

    
    use_pin_memory = (device.type == 'cuda')
    batch_size = 32 
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    
    best_val_acc = -float('inf')
    best_model_state = None

    # Train/Evaluate 5 times
    for run_idx in range(5):
        print(f"\n=== Training Run {run_idx+1} of 5 ===")

        
        model = get_resnet_model(num_classes=4, input_channels=7)  
        model.to(device)

        criterion = nn.CrossEntropyLoss()

        learning_rate = 0.0038825206157311557
        weight_decay  = 0.0001931053552153856
        gamma_rate    = 0.9388047294838997

        
        optimizer = optim.SGD(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
            momentum=0.9
        )

        scheduler = optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=gamma_rate)

        
        num_epochs = 25
        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=num_epochs
        )

        # Evaluate on validation set
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        print(f"Run {run_idx+1} validation accuracy: {val_acc:.2f}%")

        # Keep track of best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            print(f"New best model found with val_acc={val_acc:.2f}% (Run {run_idx+1}).")

    # After all 5 runs, save only the best model
    if best_model_state is not None:
        torch.save(best_model_state, MODEL_PATH)
        print(f"\nBest model saved with val_acc={best_val_acc:.2f}%")

        best_model = get_resnet_model(num_classes=4, input_channels=7)
        best_model.load_state_dict(best_model_state)
        best_model.to(device)

        test_loss, test_acc = evaluate_model(best_model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}% for the best model")

if __name__ == "__main__":
    main()

## 6. Generate the clinical model confusion matrix

This section reloads the best saved clinical model and evaluates it on the held-out testing dataset.

The confusion matrix compares the true clinical viral load class against the model-predicted class. This provides a compact summary of model performance across the four clinically defined categories and helps identify which classes are most frequently confused.

In [ ]:
def load_best_model(model_path, num_classes=4, input_channels=7, dropout_rate=0.243493213909431):
    """
    Load the best trained model from saved weights.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.conv1 = torch.nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = torch.nn.Sequential(
        torch.nn.Dropout(p=dropout_rate),
        torch.nn.Linear(model.fc.in_features, num_classes)
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()  

    return model, device

def plot_confusion_matrix(model, loader, device, class_names):
    """
    Generates and displays a confusion matrix for the model on the given loader.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    # Compute confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)  # no frequency bar
    ax.set_xlabel("Predicted HIV viral load", fontsize=12)
    ax.set_ylabel("True HIV viral load", fontsize=12)

    # Set tick label font to Arial explicitly
    ax.set_xticklabels(class_names, fontname='Arial')
    ax.set_yticklabels(class_names, fontname='Arial')

    plt.show()

model_path = MODEL_PATH
test_dataset_path = TEST_DIR

model, device = load_best_model(
    model_path=model_path,
    num_classes=4,
    input_channels=7,
)

test_dataset = PTDataset(root_dir=test_dataset_path, target_size=(500, 500))

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

class_names = ["Und.", "Low", "Med", "High"]

plot_confusion_matrix(model, test_loader, device, class_names)

